# 03 — Parameter recovery

Simulate datasets from configured ground truth, refit HDDM across replications, and summarize recovery (correlation, RMSE, CCC, ICC). **Long runtime:** set **`FIGURES_ONLY = True`** (default) to load precomputed CSVs from `results/parameter_recovery/` and skip HDDM refitting.

### Parameter recovery (configurable model)

Configure **`DEPENDS_ON_RECOVERY`**, **`INCLUDE_RECOVERY`**, and **`RECOVERY_REFERENCE_STEM` / `RECOVERY_RANGE_EXPAND`**. Ground-truth simulation ranges are derived from the reference model posterior.

**Runtime:** each rep refits HDDM on the full table — many reps × hierarchical × long MCMC can take **days**. Set **`FIGURES_ONLY = False`** locally to rerun recovery (requires a local `.nc` fit). Use `REPS_N_JOBS` for parallel reps (chain `parallel` off when `REPS_N_JOBS>1`). Checkpoint: `parameter_recovery_partial_checkpoint.csv`.


In [1]:
import os
from pathlib import Path
import sys

def _resolve_hddm_root() -> Path:
    """Locate hddm_analysis/ regardless of notebook cwd."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "trials_hddm_ready.tsv").is_file():
            return candidate
        nested = candidate / "hddm_analysis"
        if (nested / "data" / "trials_hddm_ready.tsv").is_file():
            return nested
    if start.name == "notebooks" and (start.parent / "data").is_dir():
        return start.parent
    raise FileNotFoundError(
        "Could not locate hddm_analysis package root "
        "(expected data/trials_hddm_ready.tsv). "
        f"Started search from: {start}"
    )


ROOT = str(_resolve_hddm_root())
SCRIPTS_DIR = os.path.join(ROOT, "scripts")
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
from nc_io import from_netcdf as load_idata
DATA_DIR = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
MODEL_DIR = os.path.join(ROOT, "models", "transition_prob_duration_10000samples")
FIG_DIR = os.path.join(ROOT, "figures")
for _d in (DATA_DIR, RESULTS_DIR, MODEL_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)


In [2]:
# --- Parameter recovery (configurable HDDM: depends_on / include / sim ranges) ---
import os
import numpy as np
import pandas as pd
from scipy import stats

# =============================================================================
# EDIT THIS BLOCK ONLY — add/remove parameters or change condition structure
# =============================================================================
# Default: hddm_va_vat — only v,a on conditions; t scalar in include (suffix auto: va_vat)
DEPENDS_ON_RECOVERY = {
"v": ["transition", "prob", "long_short"],
"a": ["transition", "prob", "long_short"],
}
INCLUDE_RECOVERY = ["v", "a", "t"]

# Example — hddm_va_vatz → suffix **va_vatz** (t,z scalar); add "z" to INCLUDE_RECOVERY and set
# RECOVERY_REFERENCE_STEM to the matching real-data fit (z bounds come from that posterior).
# DEPENDS_ON_RECOVERY = {"v": [...], "a": [...]}
# INCLUDE_RECOVERY = ["v", "a", "t", "z"]

# Example — all v,a,t on conditions (suffix **vat_vat**)
# DEPENDS_ON_RECOVERY = {"v": [...], "a": [...], "t": [...]}
# INCLUDE_RECOVERY = ["v", "a", "t"]

# --- Ground-truth simulation bounds from real-data posterior (then expand outward ~20–30%) ---
# Uses the same trace naming as recovery (v(cell), a(cell), scalar t, etc.).
# Reference-model .nc fits live under MODEL_DIR (models/transition_prob_duration_10000samples/).
OUT_BASE = MODEL_DIR
# Basename of your real-data fit; file = {RECOVERY_REFERENCE_STEM}_combined.nc (must match depends_on/include).
RECOVERY_REFERENCE_STEM = "hddm_va_vat"
RECOVERY_RANGE_EXPAND = 0.2  # margin = expand × (posterior max − min) on each side; try 0.2–0.3


def _posterior_var_names_for_param(par, depends_on, all_names):
    if par in depends_on:
        return [n for n in all_names if n.startswith(f"{par}(")]
    selected = [n for n in all_names if n == par]
    if not selected:
        selected = [n for n in all_names if str(n).startswith(par) and "(" not in str(n)]
    if not selected:
        selected = [n for n in all_names if n.startswith(f"{par}(")]
    return selected


def _true_uniform_ranges_from_posterior(idata, include, depends_on, expand):
    """Min/max over all matching posterior nodes per param; expand interval by expand × span."""
    posterior = idata.posterior
    all_names = [str(v) for v in posterior.data_vars]
    ranges, raw_mm = {}, {}
    for par in include:
        selected = _posterior_var_names_for_param(par, depends_on, all_names)
        if not selected:
            raise ValueError(
                f"No posterior variables for '{par}'. Check RECOVERY_REFERENCE_STEM / .nc vs INCLUDE_RECOVERY. "
                f"Sample names: {all_names[:40]}"
            )
        chunks = []
        for n in selected:
            x = np.asarray(posterior[n].values, dtype=float).ravel()
            x = x[np.isfinite(x)]
            if x.size:
                chunks.append(x)
        if not chunks:
            raise ValueError(f"No finite posterior samples for '{par}' (vars {selected})")
        samples = np.concatenate(chunks)
        lo, hi = float(np.min(samples)), float(np.max(samples))
        raw_mm[par] = (lo, hi)
        span = hi - lo
        if span <= 0:
            pad = max(abs(lo), 1e-6) * expand
            lo, hi = lo - pad, hi + pad
        else:
            m = span * expand
            lo, hi = lo - m, hi + m
        ranges[par] = (lo, hi)
    return ranges, raw_mm


IS_GROUP_MODEL = True

# Load precomputed CSVs and skip HDDM recovery (hours/days)
FIGURES_ONLY = True
PRECOMPUTED_RECOVERY_TAG = "va_vat"  # results/parameter_recovery/<tag>/

# Set True for a quick smoke test (N_SAMPLES=100, BURN=20).
TEST_RUN = False

N_REPS = 30
N_SAMPLES = 10000
BURN = 2000
N_CHAINS = 4
CHAIN_PARALLEL = True
RECOVERY_SEED = 12345
REPS_N_JOBS = max(1, min(N_REPS, os.cpu_count()))

if TEST_RUN:
    N_SAMPLES = 100
    BURN = 20

# Keep notebook output quiet to reduce document/memory pressure.
QUIET_RECOVERY_LOG = True
CHECKPOINT_EVERY_REPS = 5  # write partial CSV every k reps (or at final rep)

_RECOVERY_PARAM_ORDER = ("v", "a", "t", "z")


def recovery_file_tag(depends_on: dict, include: list) -> str:
    """Suffix like va_vat / va_vatz from letters in depends_on vs include (order v,a,t,z)."""
    unk = (set(depends_on) | set(include)) - set(_RECOVERY_PARAM_ORDER)
    if unk:
        raise ValueError(
            f"Unknown param name(s) {unk} for auto tag; extend _RECOVERY_PARAM_ORDER or rename."
        )
    dep = "".join(p for p in _RECOVERY_PARAM_ORDER if p in depends_on)
    inc = "".join(p for p in _RECOVERY_PARAM_ORDER if p in include)
    return f"{dep}_{inc}"


RECOVERY_FILE_TAG = recovery_file_tag(DEPENDS_ON_RECOVERY, INCLUDE_RECOVERY)
if FIGURES_ONLY:
    OUTPUT_TAG = PRECOMPUTED_RECOVERY_TAG
else:
    OUTPUT_TAG = (
        f"{RECOVERY_FILE_TAG}_test_s{N_SAMPLES}_b{BURN}"
        if TEST_RUN
        else RECOVERY_FILE_TAG
    )
OUT_DIR = os.path.join(RESULTS_DIR, "parameter_recovery", OUTPUT_TAG)
os.makedirs(OUT_DIR, exist_ok=True)
if TEST_RUN and not FIGURES_ONLY:
    print(f"TEST_RUN=True → outputs under {OUT_DIR} (production tag {RECOVERY_FILE_TAG} unchanged)")
PARTIAL_CSV = os.path.join(
    OUT_DIR, f"parameter_recovery_partial_{OUTPUT_TAG}.csv"
)

if FIGURES_ONLY:
    rec_path = os.path.join(OUT_DIR, f"parameter_recovery_{OUTPUT_TAG}_by_rep.csv")
    if not os.path.isfile(rec_path):
        raise FileNotFoundError(
            f"FIGURES_ONLY: missing precomputed CSV:\n  {rec_path}\n"
            "Set FIGURES_ONLY=False and provide a reference .nc to rerun recovery locally."
        )
    rec_df = pd.read_csv(rec_path, encoding="utf-8-sig")
    globals()["RECOVERY_LAST_OUT_DIR"] = OUT_DIR
    globals()["RECOVERY_LAST_FILE_TAG"] = OUTPUT_TAG
    print(f"FIGURES_ONLY=True → loaded {rec_path} ({rec_df.shape[0]} rows)")
    print(f"  Metrics/figures use: {OUT_DIR}")
else:
    import shutil
    import tempfile
    import time
    from scipy import stats
    from joblib import Parallel, delayed
    import arviz as az

    import hddm
    from hddm.generate import gen_rts

    _ref_nc = os.path.join(OUT_BASE, f"{RECOVERY_REFERENCE_STEM}_combined.nc")
    if not os.path.isfile(_ref_nc):
        _fallback_nc = os.path.join(MODEL_DIR, f"{RECOVERY_REFERENCE_STEM}_combined.nc")
        if os.path.isfile(_fallback_nc):
            _ref_nc = _fallback_nc
            OUT_BASE = MODEL_DIR
        else:
            raise FileNotFoundError(
                f"Reference posterior not found:\n  {_ref_nc}\n"
                f"  {_fallback_nc}\n"
                "Expected reference fit under MODEL_DIR (see README)."
            )
    _id_ref = load_idata(_ref_nc)
    TRUE_UNIFORM_RANGES, _RECOVERY_POSTERIOR_RAW_MINMAX = _true_uniform_ranges_from_posterior(
        _id_ref, INCLUDE_RECOVERY, DEPENDS_ON_RECOVERY, RECOVERY_RANGE_EXPAND
    )
    print(f"  Reference posterior: {_ref_nc}")
    for _p in INCLUDE_RECOVERY:
        _a, _b = _RECOVERY_POSTERIOR_RAW_MINMAX[_p]
        _lo, _hi = TRUE_UNIFORM_RANGES[_p]
        print(
            f"  {_p}: posterior min/max = [{_a:.4f}, {_b:.4f}] → TRUE_UNIFORM_RANGES [{_lo:.4f}, {_hi:.4f}] "
            f"(expand={RECOVERY_RANGE_EXPAND})"
        )
    assert set(DEPENDS_ON_RECOVERY).issubset(
        set(INCLUDE_RECOVERY)
    ), "depends_on keys must be a subset of include"
    for _p in INCLUDE_RECOVERY:
        assert _p in TRUE_UNIFORM_RANGES, f"Add TRUE_UNIFORM_RANGES entry for '{_p}'"


def discover_cond_str_fn(hddm_df, idata=None):
    """Build row -> 'cell' string consistent with HDDM trace names (e.g. Long.prob_same.flexibility)."""
    cols = ["long_short", "prob", "transition"]
    sample = hddm_df.iloc[0]
    order = cols
    if idata is not None and hasattr(idata, "posterior"):
        vkey = next(
            (str(k) for k in idata.posterior.data_vars if str(k).startswith("v(")),
            None,
        )
        if vkey is not None:
            inner = vkey.split("(", 1)[1].rstrip(")").split(".")
            order = []
            used = set()
            for tok in inner:
                for c in cols:
                    if c in used:
                        continue
                    if str(sample[c]) == tok:
                        order.append(c)
                        used.add(c)
                        break
            if len(order) != 3:
                order = cols

    def cond_str(row):
        return ".".join(str(row[c]) for c in order)

    return cond_str, order


def sample_ground_truth(cond_list, rng):
    """Return dict: param -> per-cell dict OR scalar float (if not in depends_on)."""
    true = {}
    for par in INCLUDE_RECOVERY:
        lo, hi = TRUE_UNIFORM_RANGES[par]
        if par in DEPENDS_ON_RECOVERY:
            true[par] = {c: float(rng.uniform(lo, hi)) for c in cond_list}
        else:
            true[par] = float(rng.uniform(lo, hi))
    return true


def simulate_hddm_df(template_df, true_params, cond_fn):
    """Simulate RT/response using gen_rts for all INCLUDE parameters."""
    rts, resps = [], []
    for _, row in template_df.iterrows():
        c = cond_fn(row)
        kw = {"sv": 0, "sz": 0, "st": 0}
        for par in INCLUDE_RECOVERY:
            if par in DEPENDS_ON_RECOVERY:
                kw[par] = true_params[par][c]
            else:
                kw[par] = true_params[par]
        d = gen_rts(size=1, method="cdf", **kw)
        rts.append(float(d["rt"].iloc[0]))
        resps.append(int(d["response"].iloc[0]))
    sim = template_df.copy()
    sim["rt"] = rts
    sim["response"] = resps
    return hddm.utils.flip_errors(sim)


def posterior_means_from_traces(traces_df):
    return traces_df.mean(axis=0)


def trace_scalar_group_mean(pm, basename):
    """Posterior mean for a scalar node (e.g. t, z) or mean over matching subj nodes."""
    cols = [
    k
    for k in pm.index
    if k == basename or (str(k).startswith(basename) and "(" not in str(k))
    ]
    if not cols:
        cols = [k for k in pm.index if str(k).startswith(f"{basename}(")]
        if not cols:
            raise KeyError(
        f"No trace for '{basename}'. Sample columns: {list(pm.index)[:40]}"
        )
        return float(np.mean([pm[k] for k in cols]))


def recovery_one_rep(
    template_df,
    cond_fn,
    cond_list,
    rng,
    chain_parallel=None,
    n_samples=None,
    burn=None,
    n_chains=None,
    depends_on=None,
    include=None,
    is_group_model=None,
    p_outlier=None,
):
    chain_parallel = CHAIN_PARALLEL if chain_parallel is None else chain_parallel
    n_samples = N_SAMPLES if n_samples is None else n_samples
    burn = BURN if burn is None else burn
    n_chains = N_CHAINS if n_chains is None else n_chains
    depends_on = DEPENDS_ON_RECOVERY if depends_on is None else depends_on
    include = INCLUDE_RECOVERY if include is None else include
    is_group_model = IS_GROUP_MODEL if is_group_model is None else is_group_model
    p_outlier = 0.05 if p_outlier is None else p_outlier

    true_params = sample_ground_truth(cond_list, rng)
    sim_df = simulate_hddm_df(template_df, true_params, cond_fn)

    m = hddm.HDDM(
        sim_df,
        depends_on=depends_on,
        include=include,
        is_group_model=is_group_model,
        p_outlier=p_outlier,
    )
    m.find_starting_values()
    m.sample(
        n_samples,
        burn=burn,
        chains=n_chains,
        parallel=chain_parallel,
        verbose=0,
    )
    pm = posterior_means_from_traces(m.get_traces())

    tru_by_cell = {p: [] for p in depends_on}
    rec_by_cell = {p: [] for p in depends_on}
    for c in cond_list:
        for par in depends_on:
            key = f"{par}({c})"
            if key not in pm.index:
                raise KeyError(
                    f"Missing trace {key}. Sample columns: {list(pm.index)[:24]}"
                )
            tru_by_cell[par].append(true_params[par][c])
            rec_by_cell[par].append(float(pm[key]))

    scalar_true, scalar_rec = {}, {}
    for par in include:
        if par in depends_on:
            continue
        scalar_true[par] = true_params[par]
        scalar_rec[par] = trace_scalar_group_mean(pm, par)

    return true_params, tru_by_cell, rec_by_cell, scalar_true, scalar_rec


def _rows_for_rep(rep_id, cond_list, tru_by_cell, rec_by_cell, scalar_true, scalar_rec, include, depends_on):
    """Wide table: one row per design cell + one row per scalar param (__par__)."""
    rows = []
    for i, c in enumerate(cond_list):
        row = {"rep": rep_id, "cell": c}
        for par in include:
            tc, rc = f"true_{par}", f"rec_{par}"
            if par in depends_on:
                row[tc] = tru_by_cell[par][i]
                row[rc] = rec_by_cell[par][i]
            else:
                row[tc] = np.nan
                row[rc] = np.nan
        rows.append(row)

    for par in include:
        if par in depends_on:
            continue
        row = {"rep": rep_id, "cell": f"__{par}__"}
        for p2 in include:
            tc, rc = f"true_{p2}", f"rec_{p2}"
            if p2 == par:
                row[tc] = scalar_true[par]
                row[rc] = scalar_rec[par]
            else:
                row[tc] = np.nan
                row[rc] = np.nan
        rows.append(row)
    return rows


def _recovery_rep_worker(
rep_id,
template_df,
cond_fn,
cond_list,
chain_parallel,
n_samples,
burn,
n_chains,
depends_on,
include,
is_group_model,
p_outlier,
seed_base,
):
    """One recovery repetition; isolated cwd for kabuki tmp_chain*.db files."""
    old_cwd = os.getcwd()
    workdir = tempfile.mkdtemp(prefix=f"hddm_recovery_rep{rep_id}_")
    try:
        os.chdir(workdir)
        t0 = time.perf_counter()
        rng = np.random.default_rng(seed_base + rep_id)
        _, tru_bc, rec_bc, s_true, s_rec = recovery_one_rep(
        template_df,
        cond_fn,
        cond_list,
        rng,
        chain_parallel=chain_parallel,
        n_samples=n_samples,
        burn=burn,
        n_chains=n_chains,
        depends_on=depends_on,
        include=include,
        is_group_model=is_group_model,
        p_outlier=p_outlier,
        )
        rows = _rows_for_rep(
        rep_id,
        cond_list,
        tru_bc,
        rec_bc,
        s_true,
        s_rec,
        include,
        depends_on,
        )
        elapsed = time.perf_counter() - t0
        return rep_id, rows, elapsed
    finally:
        try:
            os.chdir(old_cwd)
        except OSError:
            pass
        shutil.rmtree(workdir, ignore_errors=True)


def flip_errors(data: pd.DataFrame) -> pd.DataFrame:
    if np.any(data["rt"] < 0):
        return data
    out = data.copy()
    out.loc[out["response"] != 1, "rt"] = -out.loc[out["response"] != 1, "rt"]
    return out


def _write_progress(done, total, mode, extra=""):
    pct = 100.0 * done / max(total, 1)
    with open(progress_path, "w", encoding="utf-8") as f:
        f.write(
            f"done={done}/{total} ({pct:.1f}%) | mode={mode} | tag={OUTPUT_TAG} | out={OUT_DIR} {extra}".strip()
        )


if not FIGURES_ONLY:
    # Load HDDM-ready data from bundled TSV.
    if "hddm_data" not in globals():
        trials = pd.read_csv(os.path.join(DATA_DIR, "trials_hddm_ready.tsv"), sep="\t")
        hddm_data = trials.copy()
        hddm_data["response"] = hddm_data["acc"].astype(int)
        hddm_data["rt"] = hddm_data["rt_ms"] / 1000.0
        hddm_data = hddm_data[
            ["subj_idx", "transition", "prob", "long_short", "response", "rt"]
        ]
        hddm_data = flip_errors(hddm_data)
        print("Loaded hddm_data:", hddm_data.shape)
    else:
        print("Using existing hddm_data:", hddm_data.shape)

    _idata_hint = globals().get("idata", None)
    cond_fn, _order = discover_cond_str_fn(hddm_data, _idata_hint)
    cond_list = sorted(hddm_data.apply(cond_fn, axis=1).unique())

    chain_parallel_effective = CHAIN_PARALLEL if REPS_N_JOBS == 1 else False
    progress_path = os.path.join(OUT_DIR, f"parameter_recovery_progress_{OUTPUT_TAG}.txt")

    _write_progress(0, N_REPS, "sequential" if REPS_N_JOBS <= 1 else "parallel")

    if REPS_N_JOBS <= 1:
        rng = np.random.default_rng(RECOVERY_SEED)
        all_rows = []
        for rep in range(N_REPS):
            _, tru_bc, rec_bc, s_true, s_rec = recovery_one_rep(
                hddm_data,
                cond_fn,
                cond_list,
                rng,
                chain_parallel=chain_parallel_effective,
            )
            all_rows.extend(
                _rows_for_rep(
                    rep, cond_list, tru_bc, rec_bc, s_true, s_rec, INCLUDE_RECOVERY, DEPENDS_ON_RECOVERY
                )
            )
            if (rep + 1) % CHECKPOINT_EVERY_REPS == 0 or (rep + 1) == N_REPS:
                pd.DataFrame(all_rows).to_csv(PARTIAL_CSV, index=False, encoding="utf-8-sig")
                _write_progress(rep + 1, N_REPS, "sequential", extra=f"| rows={len(all_rows)}")
    else:
        results = Parallel(n_jobs=REPS_N_JOBS, backend="loky", verbose=0)(
            delayed(_recovery_rep_worker)(
                rep,
                hddm_data,
                cond_fn,
                cond_list,
                chain_parallel_effective,
                N_SAMPLES,
                BURN,
                N_CHAINS,
                DEPENDS_ON_RECOVERY,
                INCLUDE_RECOVERY,
                IS_GROUP_MODEL,
                0.05,
                RECOVERY_SEED,
            )
            for rep in range(N_REPS)
        )
        results = sorted(results, key=lambda x: x[0])
        all_rows = []
        for rep_id, rows, _elapsed in results:
            all_rows.extend(rows)
        pd.DataFrame(all_rows).to_csv(PARTIAL_CSV, index=False, encoding="utf-8-sig")
        _write_progress(N_REPS, N_REPS, "parallel", extra=f"| rows={len(all_rows)}")

    rec_df = pd.DataFrame(all_rows)
    rec_path = os.path.join(OUT_DIR, f"parameter_recovery_{OUTPUT_TAG}_by_rep.csv")
    rec_df.to_csv(rec_path, index=False, encoding="utf-8-sig")
    if not QUIET_RECOVERY_LOG:
        print("Saved:", rec_path)
        print("Progress file:", progress_path)

scalar_cells = [f"__{p}__" for p in INCLUDE_RECOVERY if p not in DEPENDS_ON_RECOVERY]
cell_df = rec_df[~rec_df["cell"].isin(scalar_cells)].copy()


def corr_report(x, y, name):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return name, np.nan, np.nan
    r, p = stats.pearsonr(x[mask], y[mask])
    print(f"{name}: Pearson r = {r:.3f}, p = {p:.2e}, n = {mask.sum()}")
    return name, r, p


summary = []
for par in INCLUDE_RECOVERY:
    tx, ty = f"true_{par}", f"rec_{par}"
    if par in DEPENDS_ON_RECOVERY:
        summary.append(
            corr_report(cell_df[tx], cell_df[ty], f"{par} (cells × reps)")
        )
    else:
        sdf = rec_df[rec_df["cell"] == f"__{par}__"]
        summary.append(corr_report(sdf[tx], sdf[ty], f"{par} (scalar per rep)"))

summary_df = pd.DataFrame(summary, columns=["parameter", "pearson_r", "p_value"])
summary_df.to_csv(
    os.path.join(OUT_DIR, f"parameter_recovery_{OUTPUT_TAG}_correlations.csv"),
    index=False,
    encoding="utf-8-sig",
)

globals()["RECOVERY_LAST_OUT_DIR"] = OUT_DIR
globals()["RECOVERY_LAST_FILE_TAG"] = OUTPUT_TAG


FIGURES_ONLY=True → loaded D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\parameter_recovery\va_vat\parameter_recovery_va_vat_by_rep.csv (270 rows)
  Metrics/figures use: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\parameter_recovery\va_vat
v (cells × reps): Pearson r = 0.985, p = 3.00e-184, n = 240
a (cells × reps): Pearson r = 0.942, p = 3.23e-115, n = 240
t (scalar per rep): Pearson r = 0.924, p = 3.15e-13, n = 30


In [3]:
# Parameter recovery metrics
# 1) Per-condition (single-cell) correlations: corr(true, recovered) across reps.
# 2) Delta-score recovery: corr(Δ_true, Δ_estimated) across reps for all main effects + interactions.
#
# After running the recovery cell: uses RECOVERY_LAST_OUT_DIR / RECOVERY_LAST_FILE_TAG.
# Otherwise falls back to OUT_BASE + tag (edit RECOVERY_FILE_TAG if you run metrics alone).
import os
import itertools
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_BASE = MODEL_DIR
RECOVERY_FILE_TAG = globals().get("RECOVERY_LAST_FILE_TAG", "va_vat")
OUT_DIR = globals().get(
    "RECOVERY_LAST_OUT_DIR",
    os.path.join(RESULTS_DIR, "parameter_recovery", RECOVERY_FILE_TAG),
)
REC_CSV = os.path.join(OUT_DIR, f"parameter_recovery_{RECOVERY_FILE_TAG}_by_rep.csv")
if not os.path.isfile(REC_CSV):
    raise FileNotFoundError(
        f"Missing recovery table:\n  {REC_CSV}\n"
        "Run the previous cell with FIGURES_ONLY=True (default) or complete a local recovery run."
    )
print("Using:", REC_CSV)

# Layout presets: single-column ~89 mm, double-column ~183 mm.
MM_TO_INCH = 1.0 / 25.4
FIG_LAYOUT = "single"  # "single" or "double"
FIG_WIDTH_MM = 89 if FIG_LAYOUT == "single" else 183
FIG_BASE_W = FIG_WIDTH_MM * MM_TO_INCH
FIG_BASE_H = FIG_BASE_W * 0.72
PLOT_THEME = "nature"  # "nature" or "science"

THEME = {
    "nature": {
        "per_cell": "#3B4CC0",
        "delta": "#B40426",
        "identity": "#4D4D4D",
        "grid": "#C7C7C7",
    },
    "science": {
        "per_cell": "#1F77B4",
        "delta": "#D62728",
        "identity": "#5A5A5A",
        "grid": "#D2D2D2",
    },
}[PLOT_THEME]


def setup_pub_plot_style() -> None:
    # Plot defaults with mathtext (LaTeX-like without external TeX dependency).
    plt.rcParams.update(
        {
            "figure.dpi": 300,
            "savefig.dpi": 600,
            "savefig.bbox": "tight",
            "savefig.transparent": True,
            "savefig.facecolor": "none",
            "font.family": "serif",
            "font.size": 8.5,
            "axes.titlesize": 8.5,
            "axes.labelsize": 8.5,
            "xtick.labelsize": 7.5,
            "ytick.labelsize": 7.5,
            "legend.fontsize": 7.5,
            "axes.linewidth": 0.75,
            "xtick.major.width": 0.75,
            "ytick.major.width": 0.75,
            "xtick.major.size": 2.8,
            "ytick.major.size": 2.8,
            "mathtext.default": "regular",
        }
    )


def add_identity_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    lo = float(min(np.min(x), np.min(y)))
    hi = float(max(np.max(x), np.max(y)))

    if np.isclose(lo, hi):
        span = 1.0 if hi == 0 else abs(hi)
    else:
        span = hi - lo

    # Add a small margin so edge points/markers are not clipped by axes.
    pad = 0.06 * span
    lo_lim = lo - pad
    hi_lim = hi + pad

    ax.plot([lo_lim, hi_lim], [lo_lim, hi_lim], linestyle="--", linewidth=0.8, color=THEME["identity"], alpha=0.9)
    ax.set_xlim(lo_lim, hi_lim)
    ax.set_ylim(lo_lim, hi_lim)

    # Keep data region square in every panel.
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1)


def clean_title(text: str, width: int = 32) -> str:
    return "\n".join(textwrap.wrap(str(text), width=width, break_long_words=False))


def _to_perturbation_level(v: str) -> str:
    m = {
        "Long": "Low Flux",
        "Short": "High Flux",
    }
    return m.get(str(v), str(v))


def _to_predictability_level(v: str) -> str:
    m = {
        "prob_diff": "Low Pred.",
        "prob_same": "High Pred.",
    }
    return m.get(str(v), str(v))


def _to_control_level(v: str) -> str:
    m = {
        "flexibility": "Flexibility",
        "stability": "Stability",
    }
    return m.get(str(v), str(v))


def condition_label_from_cell(cell: str) -> str:
    parts = str(cell).split(".")
    if len(parts) != 3:
        return str(cell)
    pert = _to_perturbation_level(parts[0])
    pred = _to_predictability_level(parts[1])
    ctrl = _to_control_level(parts[2])
    return f"{pert}, {pred}, {ctrl}"


def effect_label_from_code(effect_code: str) -> str:
    s = str(effect_code)
    if s.startswith("main_effect_factor_1"):
        return "Main Effect: Perturbation (Low vs. High)"
    if s.startswith("main_effect_factor_2"):
        return "Main Effect: Predictability (Low vs. High)"
    if s.startswith("main_effect_factor_3"):
        return "Main Effect: Control (Flexibility vs. Stability)"
    if s.startswith("interaction_factor_1_factor_2_factor_3"):
        return "3-Way Int.: Perturbation x Predictability x Control"
    if s.startswith("interaction_factor_1_factor_2"):
        return "2-Way Int.: Perturbation x Predictability"
    if s.startswith("interaction_factor_1_factor_3"):
        return "2-Way Int.: Perturbation x Control"
    if s.startswith("interaction_factor_2_factor_3"):
        return "2-Way Int.: Predictability x Control"
    return s


def seven_panel_axes(fig):
    # Exact hexagonal layout: six outer panels at 60-degree intervals + one center panel.
    # Returns axes in order [top, upper-right, lower-right, bottom, lower-left, upper-left, center].
    # Spacing control: smaller horizontal gap, larger vertical gap.
    panel_w = 0.232
    panel_h = 0.232
    cx, cy = 0.5, 0.5
    rx = 0.285  # horizontal radius (tighter)
    ry = 0.331  # vertical radius (gap ~= 2/3 of previous)

    outer_axes = []
    angles_deg = [90, 30, -30, -90, -150, 150]
    for deg in angles_deg:
        rad = np.deg2rad(deg)
        x_center = cx + rx * np.cos(rad)
        y_center = cy + ry * np.sin(rad)
        left = x_center - panel_w / 2
        bottom = y_center - panel_h / 2
        outer_axes.append(fig.add_axes([left, bottom, panel_w, panel_h]))

    center_ax = fig.add_axes([cx - panel_w / 2, cy - panel_h / 2, panel_w, panel_h])
    return outer_axes + [center_ax]


def squeeze_grid_horizontally(axes, nrow: int, ncol: int, gap_scale: float) -> None:
    """Compress the horizontal gaps between subplot *frames*.

    gap_scale < 1 shrinks inter-column gaps while preserving relative layout.
    This works even when wspace is effectively overridden by label extents.
    """
    axes = list(axes)
    if ncol < 2:
        return

    rep_row = 0
    cols = [axes[rep_row * ncol + c] for c in range(ncol) if (rep_row * ncol + c) < len(axes)]
    pos = [ax.get_position() for ax in cols]
    lefts = [p.x0 for p in pos]
    rights = [p.x1 for p in pos]

    gaps = [max(0.0, lefts[i + 1] - rights[i]) for i in range(len(cols) - 1)]
    new_gaps = [g * gap_scale for g in gaps]

    target_lefts = [lefts[0]]
    for i in range(len(cols) - 1):
        w_i = rights[i] - lefts[i]
        target_lefts.append(target_lefts[-1] + w_i + new_gaps[i])

    for c in range(ncol):
        if c >= len(target_lefts):
            continue
        dx = target_lefts[c] - lefts[c]
        for r in range(nrow):
            idx = r * ncol + c
            if idx >= len(axes):
                continue
            p = axes[idx].get_position()
            axes[idx].set_position([p.x0 + dx, p.y0, p.width, p.height])


def expand_grid_vertically(axes, nrow: int, ncol: int, gap_multiplier: float) -> None:
    """Increase the vertical gaps between rows by a multiplier.

    To keep the overall figure span the same, row heights are scaled down as needed.
    This is robust when hspace has little visible effect.
    """
    axes = list(axes)
    if nrow < 2:
        return

    # Use first column to infer row geometry.
    col0 = []
    for r in range(nrow):
        idx = r * ncol
        if idx < len(axes):
            col0.append(axes[idx])

    if len(col0) < 2:
        return

    pos = [ax.get_position() for ax in col0]
    # sort rows bottom->top
    order = sorted(range(len(pos)), key=lambda i: pos[i].y0)
    pos = [pos[i] for i in order]

    bottoms = [p.y0 for p in pos]
    tops = [p.y1 for p in pos]
    heights = [t - b for t, b in zip(tops, bottoms)]
    gaps = [max(0.0, bottoms[i + 1] - tops[i]) for i in range(len(pos) - 1)]

    span_bottom = bottoms[0]
    span_top = tops[-1]
    total_span = span_top - span_bottom

    new_gaps = [g * gap_multiplier for g in gaps]
    sum_new_gaps = sum(new_gaps)
    sum_heights = sum(heights)

    if total_span <= 0:
        return

    # scale heights down if needed to fit increased gaps
    if sum_new_gaps >= total_span:
        # extreme: cap gaps to 80% of span
        factor = (0.80 * total_span) / max(sum_new_gaps, 1e-9)
        new_gaps = [g * factor for g in new_gaps]
        sum_new_gaps = sum(new_gaps)

    height_scale = (total_span - sum_new_gaps) / max(sum_heights, 1e-9)
    height_scale = max(0.30, min(1.0, height_scale))

    new_heights = [h * height_scale for h in heights]

    # Compute new bottoms by stacking rows bottom->top
    new_bottoms = [span_bottom]
    for i in range(len(new_heights) - 1):
        new_bottoms.append(new_bottoms[-1] + new_heights[i] + new_gaps[i])

    # Apply per-row y transform to every column
    for r_vis, (old_b, old_h, new_b, new_h) in enumerate(zip(bottoms, heights, new_bottoms, new_heights)):
        dy = new_b - old_b
        hs = new_h / max(old_h, 1e-9)
        # affect all columns in that row (original row index in the grid = order[r_vis])
        r = order[r_vis]
        for c in range(ncol):
            idx = r * ncol + c
            if idx >= len(axes):
                continue
            p = axes[idx].get_position()
            axes[idx].set_position([p.x0, p.y0 + dy, p.width, p.height * hs])


def save_pub_figure(fig, out_base: str) -> None:
    """Export one figure to PNG (600 dpi), PDF, EPS, and SVG."""
    base_kw = dict(bbox_inches="tight", facecolor="none", edgecolor="none")
    for ext in ("png", "pdf", "eps", "svg"):
        path = f"{out_base}.{ext}"
        if ext == "png":
            fig.savefig(path, transparent=True, dpi=600, **base_kw)
        elif ext == "eps":
            # PostScript backend does not support transparency.
            fig.savefig(path, format="eps", **base_kw)
        else:
            fig.savefig(path, transparent=True, **base_kw)
        print("Saved:", path)


def pearson_r(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    if int(m.sum()) < 3:
        return float("nan")
    xc = x[m] - float(np.mean(x[m]))
    yc = y[m] - float(np.mean(y[m]))
    denom = float(np.sqrt(np.sum(xc**2) * np.sum(yc**2)))
    if denom <= 0 or not np.isfinite(denom):
        return float("nan")
    return float(np.sum(xc * yc) / denom)


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return float("nan")
    e = y_true[mask].astype(float) - y_pred[mask].astype(float)
    return float(np.sqrt(np.mean(e**2)))


def true_value_range(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    x = y_true[mask].astype(float)
    if len(x) < 1:
        return float("nan")
    return float(np.ptp(x))


def nrmse_percent(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    rng = true_value_range(y_true, y_pred)
    if not np.isfinite(rng) or rng <= 0:
        return float("nan")
    return float(100.0 * rmse(y_true, y_pred) / rng)


def lin_ccc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    x = y_true[mask].astype(float)
    y = y_pred[mask].astype(float)
    if len(x) < 2:
        return float("nan")
    mx, my = np.mean(x), np.mean(y)
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    cov_xy = np.cov(x, y, ddof=1)[0, 1]
    denom = vx + vy + (mx - my) ** 2
    if denom <= 0 or not np.isfinite(denom):
        return float("nan")
    return float(2 * cov_xy / denom)


def icc_3_1_two_raters(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    a = y_true[mask].astype(float)
    b = y_pred[mask].astype(float)
    n = len(a)
    if n < 2:
        return float("nan")
    k = 2
    Y = np.column_stack([a, b])
    grand = float(np.mean(Y))
    SS_total = float(np.sum((Y - grand) ** 2))
    row_mean = Y.mean(axis=1)
    col_mean = Y.mean(axis=0)
    SS_row = float(k * np.sum((row_mean - grand) ** 2))
    SS_col = float(n * np.sum((col_mean - grand) ** 2))
    SS_e = SS_total - SS_row - SS_col
    df_row = n - 1
    df_e = (n - 1) * (k - 1)
    if df_e <= 0:
        return float("nan")
    MSR = SS_row / df_row
    MSE = SS_e / df_e
    denom = MSR + (k - 1) * MSE
    if denom == 0 or not np.isfinite(denom):
        return float("nan")
    return float((MSR - MSE) / denom)


df = pd.read_csv(REC_CSV, encoding="utf-8-sig")
true_cols = [c for c in df.columns if c.startswith("true_")]
params = [c.replace("true_", "") for c in true_cols]
mask_scalar = df["cell"].astype(str).str.match(r"^__\w+__$")
mask_cell = ~mask_scalar
cell_df = df.loc[mask_cell].copy()

# -----------------------------------------------------------------------------
# 1) Per-condition recovery metrics (report these first)
# -----------------------------------------------------------------------------
per_cell_rows = []
per_cell_points = []
for par in params:
    tc, rc = f"true_{par}", f"rec_{par}"

    # scalar parameter: one value per rep
    if (df["cell"] == f"__{par}__").any():
        sdf = df.loc[df["cell"] == f"__{par}__", [tc, rc]].copy()
        yt = sdf[tc].to_numpy(dtype=float)
        yp = sdf[rc].to_numpy(dtype=float)
        msk = np.isfinite(yt) & np.isfinite(yp)
        per_cell_rows.append(
            {
                "parameter": par,
                "cell": f"__{par}__",
                "n_pairs": int(msk.sum()),
                "true_range": true_value_range(yt, yp),
                "pearson_r": pearson_r(yt, yp),
                "RMSE": rmse(yt, yp),
                "nRMSE_pct": nrmse_percent(yt, yp),
                "CCC": lin_ccc(yt, yp),
                "ICC3_1": icc_3_1_two_raters(yt, yp),
            }
        )
        for xv, yv in zip(yt, yp):
            if np.isfinite(xv) and np.isfinite(yv):
                per_cell_points.append({"parameter": par, "cell": f"__{par}__", "x": float(xv), "y": float(yv)})
        continue

    # cell-varying parameter: compute metrics within each condition across reps
    for c in sorted(cell_df["cell"].unique()):
        sdf = cell_df.loc[cell_df["cell"] == c, [tc, rc]].copy()
        yt = sdf[tc].to_numpy(dtype=float)
        yp = sdf[rc].to_numpy(dtype=float)
        msk = np.isfinite(yt) & np.isfinite(yp)
        per_cell_rows.append(
            {
                "parameter": par,
                "cell": c,
                "n_pairs": int(msk.sum()),
                "true_range": true_value_range(yt, yp),
                "pearson_r": pearson_r(yt, yp),
                "RMSE": rmse(yt, yp),
                "nRMSE_pct": nrmse_percent(yt, yp),
                "CCC": lin_ccc(yt, yp),
                "ICC3_1": icc_3_1_two_raters(yt, yp),
            }
        )
        for xv, yv in zip(yt, yp):
            if np.isfinite(xv) and np.isfinite(yv):
                per_cell_points.append({"parameter": par, "cell": c, "x": float(xv), "y": float(yv)})

per_cell_corr_df = pd.DataFrame(per_cell_rows)
per_cell_out_csv = os.path.join(OUT_DIR, f"parameter_recovery_{RECOVERY_FILE_TAG}_per_cell_correlations.csv")
per_cell_corr_df.to_csv(per_cell_out_csv, index=False, encoding="utf-8-sig")
print("Saved:", per_cell_out_csv)
display(per_cell_corr_df.sort_values(["parameter", "cell"]))

# Per-cell scatter plots: 17 panels in a polygon layout
# Center: t recovery (scalar per rep). Outer ring: 16 condition panels (v and a), arranged symmetrically.
setup_pub_plot_style()
per_cell_points_df = pd.DataFrame(per_cell_points)


def _split_two_rings(n_total: int):
    # Heuristic split: small inner ring for readability, remainder outer ring.
    # Keep inner ring >= 5 (looks like a ring), unless n_total is small.
    if n_total <= 9:
        return max(3, n_total // 2), n_total - max(3, n_total // 2)
    inner = int(round(n_total * 0.35))
    inner = max(5, min(inner, n_total - 5))
    outer = n_total - inner
    return inner, outer


def ring_panel_axes_two_rings(
    fig,
    n_inner: int,
    n_outer: int,
    panel_w: float,
    panel_h: float,
    cx: float = 0.5,
    cy: float = 0.5,
    r_inner_x: float = 0.24,
    r_inner_y: float = 0.24,
    r_outer_x: float = 0.42,
    r_outer_y: float = 0.42,
):
    def _axes_for_ring(n, rx, ry):
        angles = np.linspace(90, 90 - 360, n, endpoint=False)
        out = []
        for deg in angles:
            rad = np.deg2rad(deg)
            x_center = cx + rx * np.cos(rad)
            y_center = cy + ry * np.sin(rad)
            out.append(fig.add_axes([x_center - panel_w / 2, y_center - panel_h / 2, panel_w, panel_h]))
        return out

    inner_axes = _axes_for_ring(n_inner, r_inner_x, r_inner_y)
    outer_axes = _axes_for_ring(n_outer, r_outer_x, r_outer_y)
    return inner_axes, outer_axes


# Build canonical cell order (8 conditions)
cond_order = [
    "Low Flux, Low Pred., Flexibility",
    "Low Flux, Low Pred., Stability",
    "Low Flux, High Pred., Flexibility",
    "Low Flux, High Pred., Stability",
    "High Flux, Low Pred., Flexibility",
    "High Flux, Low Pred., Stability",
    "High Flux, High Pred., Flexibility",
    "High Flux, High Pred., Stability",
]

# Map existing cell strings to canonical labels
if not per_cell_points_df.empty:
    per_cell_points_df["cell_label"] = per_cell_points_df["cell"].apply(condition_label_from_cell)

# Extract t scalar points
t_points = per_cell_points_df[(per_cell_points_df["parameter"] == "t") & (per_cell_points_df["cell"].astype(str) == "__t__")].copy()

# Extract v/a condition points (8 each)
va_points = per_cell_points_df[per_cell_points_df["parameter"].isin(["v", "a"])].copy()

# Two-ring layout (auto split; no overlap)
# --- Two-ring per-cell layout (17 panels total) ---
# Requested: inner ring 6, outer ring 11.
n_inner, n_outer = 6, 11

fig_w = FIG_BASE_W * 3.8
fig_h = FIG_BASE_W * 3.8
fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)

panel_w = 0.155
panel_h = 0.155
inner_axes, outer_axes = ring_panel_axes_two_rings(
    fig,
    n_inner=n_inner,
    n_outer=n_outer,
    panel_w=panel_w,
    panel_h=panel_h,
    # inner ring pulled inward by ~1/4 panel, especially vertically
    r_inner_x=0.215,
    r_inner_y=0.200,
    r_outer_x=0.44,
    r_outer_y=0.44,
)

# Split conditions into two groups of 3/5 for inner/outer rings
inner_conds = cond_order[:3]
outer_conds = cond_order[3:]

# Split conditions to match requested ring sizes.
# Inner ring has 6 panels: t + 5 others (2 v conditions + 3 a conditions).
v_inner_conds = cond_order[:2]
a_inner_conds = cond_order[:3]

v_outer_conds = [c for c in cond_order if c not in v_inner_conds]
a_outer_conds = [c for c in cond_order if c not in a_inner_conds]

inner_specs = [("t", "__t__")] + [("v", c) for c in v_inner_conds] + [("a", c) for c in a_inner_conds]
outer_specs = [("v", c) for c in v_outer_conds] + [("a", c) for c in a_outer_conds]

# Trim to exact ring sizes
inner_specs = inner_specs[:n_inner]
outer_specs = outer_specs[:n_outer]


def _draw_per_cell(ax, par, cell_label):
    if par == "t":
        x = t_points["x"].to_numpy(dtype=float)
        y = t_points["y"].to_numpy(dtype=float)
        if len(x) > 0:
            # requirement: same color as other per-cell panels
            ax.scatter(x, y, s=18, alpha=0.78, color=THEME["per_cell"], edgecolors="white", linewidths=0.3)
            add_identity_line(ax, x, y)
            ax.grid(alpha=0.2, linewidth=0.35, color=THEME["grid"])
            rr = pearson_r(x, y)
            ax.text(
                0.02,
                0.98,
                rf"$r={rr:.2f}$, $n={len(x)}$",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=7.2,
                bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
            )
        ax.set_xlabel(r"$t_{true}$", labelpad=1)
        ax.set_ylabel(r"$\hat{t}$", labelpad=1)
        # requirement: remove small title for t
        return

    sub = va_points[(va_points["parameter"] == par) & (va_points["cell_label"] == cell_label)].copy()
    x = sub["x"].to_numpy(dtype=float)
    y = sub["y"].to_numpy(dtype=float)
    if len(x) > 0:
        ax.scatter(x, y, s=18, alpha=0.78, color=THEME["per_cell"], edgecolors="white", linewidths=0.3)
        add_identity_line(ax, x, y)
        ax.grid(alpha=0.2, linewidth=0.35, color=THEME["grid"])
        rr = pearson_r(x, y)
        ax.text(
            0.02,
            0.98,
            rf"$r={rr:.2f}$, $n={len(x)}$",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7.0,
            bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "alpha": 0.80, "edgecolor": "none"},
        )
    ax.set_xlabel(rf"${par}_{{true}}$", labelpad=1)
    ax.set_ylabel(rf"$\hat{{{par}}}$", labelpad=1)
    ax.set_title(clean_title(cell_label, width=18), pad=2)


for ax, (par, cl) in zip(inner_axes, inner_specs):
    _draw_per_cell(ax, par, cl)
for ax, (par, cl) in zip(outer_axes, outer_specs):
    _draw_per_cell(ax, par, cl)

# Hide any unused axes (prevents blank frame-only panels)
for ax in inner_axes[len(inner_specs):]:
    ax.set_visible(False)
for ax in outer_axes[len(outer_specs):]:
    ax.set_visible(False)

RECOVERY_FIG_DIR = os.path.join(FIG_DIR, "parameter_recovery", RECOVERY_FILE_TAG)
os.makedirs(RECOVERY_FIG_DIR, exist_ok=True)
out_base = os.path.join(RECOVERY_FIG_DIR, f"parameter_recovery_{RECOVERY_FILE_TAG}_per_cell_scatter_vat_two_rings")
save_pub_figure(fig, out_base)
plt.close(fig)

# -----------------------------------------------------------------------------
# 2) Delta-score recovery: corr(Δ_true, Δ_estimated) across reps
# -----------------------------------------------------------------------------
cell_parts = cell_df["cell"].astype(str).str.split(".", expand=True)
if cell_parts.shape[1] < 2:
    raise ValueError(
        "Expected 'cell' values like 'Long.prob_same.flexibility' (dot-separated). "
        f"Got example: {cell_df['cell'].iloc[0]!r}"
    )

factor_cols = [f"factor_{i+1}" for i in range(cell_parts.shape[1])]
for i, col in enumerate(factor_cols):
    cell_df[col] = cell_parts[i].astype(str)

factors_levels = {c: sorted(cell_df[c].unique()) for c in factor_cols}


def _main_effect_specs():
    specs = []
    for f, levels in factors_levels.items():
        if len(levels) < 2:
            continue
        for l1, l2 in itertools.combinations(levels, 2):
            specs.append((f, l1, l2))
    return specs


def _interaction_specs():
    specs = []
    factor_names = list(factors_levels.keys())
    for k in range(2, len(factor_names) + 1):
        for subset in itertools.combinations(factor_names, k):
            pair_lists = []
            for f in subset:
                lv = factors_levels[f]
                if len(lv) < 2:
                    pair_lists = []
                    break
                pair_lists.append(list(itertools.combinations(lv, 2)))
            if not pair_lists:
                continue
            for pairs in itertools.product(*pair_lists):
                level_pairs = {f: p for f, p in zip(subset, pairs)}
                specs.append((subset, level_pairs))
    return specs


def _main_effect_delta(df_rep: pd.DataFrame, value_col: str, f: str, l1: str, l2: str) -> float:
    remaining = [c for c in factor_cols if c != f]
    if remaining:
        groups = df_rep.groupby(remaining, dropna=False)
    else:
        groups = [(None, df_rep)]

    diffs = []
    for _, g in groups:
        v1 = g.loc[g[f] == l1, value_col].astype(float).to_numpy()
        v2 = g.loc[g[f] == l2, value_col].astype(float).to_numpy()
        v1 = v1[np.isfinite(v1)]
        v2 = v2[np.isfinite(v2)]
        if v1.size == 0 or v2.size == 0:
            continue
        diffs.append(float(np.mean(v1) - np.mean(v2)))

    if len(diffs) == 0:
        return float("nan")
    return float(np.mean(diffs))


def _interaction_contrast(df_rep: pd.DataFrame, value_col: str, subset, level_pairs) -> float:
    remaining = [c for c in factor_cols if c not in subset]
    if remaining:
        groups = df_rep.groupby(remaining, dropna=False)
    else:
        groups = [(None, df_rep)]

    contrasts = []
    for _, g in groups:
        ok = True
        s = 0.0
        for pick in itertools.product([0, 1], repeat=len(subset)):
            w = 1
            mask = np.ones(len(g), dtype=bool)
            for f, bit in zip(subset, pick):
                l1, l2 = level_pairs[f]
                lvl = l1 if bit == 0 else l2
                mask &= (g[f].to_numpy() == lvl)
                w *= (1 if bit == 0 else -1)
            if not mask.any():
                ok = False
                break
            v = g.loc[mask, value_col].astype(float).to_numpy()
            v = v[np.isfinite(v)]
            if v.size == 0:
                ok = False
                break
            s += w * float(np.mean(v))
        if ok:
            contrasts.append(s)

    if len(contrasts) == 0:
        return float("nan")
    return float(np.mean(contrasts))


reps = sorted(df["rep"].dropna().unique())
delta_rows = []
delta_points = []

main_specs = _main_effect_specs()
int_specs = _interaction_specs()

for par in params:
    tc, rc = f"true_{par}", f"rec_{par}"

    if (df["cell"] == f"__{par}__").any():
        continue

    per_rep_records = []
    for rep in reps:
        drep = cell_df.loc[cell_df["rep"] == rep, factor_cols + [tc, rc]].copy()
        if drep.empty:
            continue

        for f, l1, l2 in main_specs:
            dt = _main_effect_delta(drep, tc, f, l1, l2)
            de = _main_effect_delta(drep, rc, f, l1, l2)
            per_rep_records.append(
                {
                    "rep": rep,
                    "parameter": par,
                    "effect_type": "main_effect",
                    "effect": f"main_effect_{f}_{l1}_vs_{l2}",
                    "delta_true": dt,
                    "delta_estimated": de,
                }
            )

        for subset, level_pairs in int_specs:
            dt = _interaction_contrast(drep, tc, subset, level_pairs)
            de = _interaction_contrast(drep, rc, subset, level_pairs)
            tag = "__".join([f"{f}_{level_pairs[f][0]}_vs_{level_pairs[f][1]}" for f in subset])
            per_rep_records.append(
                {
                    "rep": rep,
                    "parameter": par,
                    "effect_type": "interaction",
                    "effect": f"interaction_{'_'.join(subset)}__{tag}",
                    "delta_true": dt,
                    "delta_estimated": de,
                }
            )

    if not per_rep_records:
        continue

    per_rep_df = pd.DataFrame(per_rep_records)
    for (etype, eff), g in per_rep_df.groupby(["effect_type", "effect"], dropna=False):
        yt = g["delta_true"].to_numpy(dtype=float)
        yp = g["delta_estimated"].to_numpy(dtype=float)
        msk = np.isfinite(yt) & np.isfinite(yp)
        delta_rows.append(
            {
                "parameter": par,
                "effect_type": etype,
                "effect": eff,
                "n_pairs": int(msk.sum()),
                "true_range": true_value_range(yt, yp),
                "pearson_r": pearson_r(yt, yp),
                "RMSE": rmse(yt, yp),
                "nRMSE_pct": nrmse_percent(yt, yp),
                "CCC": lin_ccc(yt, yp),
                "ICC3_1": icc_3_1_two_raters(yt, yp),
            }
        )
        for xv, yv in zip(yt, yp):
            if np.isfinite(xv) and np.isfinite(yv):
                delta_points.append(
                    {
                        "parameter": par,
                        "effect_type": etype,
                        "effect": eff,
                        "x": float(xv),
                        "y": float(yv),
                    }
                )

delta_corr_df = pd.DataFrame(delta_rows)
delta_out_csv = os.path.join(OUT_DIR, f"parameter_recovery_{RECOVERY_FILE_TAG}_delta_recovery_correlations.csv")
delta_corr_df.to_csv(delta_out_csv, index=False, encoding="utf-8-sig")
print("Saved:", delta_out_csv)
display(delta_corr_df.sort_values(["parameter", "effect_type", "effect"]))

# Delta-recovery scatter plots: two-ring layout (auto split; no overlap)
# Arrange content so a-panels are mostly on left half and v-panels on right half.
delta_points_df = pd.DataFrame(delta_points)


def _select_effects_for_param(dpar: pd.DataFrame):
    ordered = [
        "main_effect_factor_1",
        "main_effect_factor_2",
        "main_effect_factor_3",
        "interaction_factor_1_factor_2",
        "interaction_factor_1_factor_3",
        "interaction_factor_2_factor_3",
        "interaction_factor_1_factor_2_factor_3",
    ]
    effect_lookup = list(dpar["effect"].unique())
    selected = []
    for key in ordered:
        match = next((e for e in effect_lookup if str(e).startswith(key)), None)
        if match is not None:
            selected.append(match)
    for e in sorted(effect_lookup):
        if e not in selected:
            selected.append(e)
    return selected[:7]


if not delta_points_df.empty:
    dv = delta_points_df[delta_points_df["parameter"] == "v"].copy()
    da = delta_points_df[delta_points_df["parameter"] == "a"].copy()

    eff_keys = [
        "main_effect_factor_1",
        "main_effect_factor_2",
        "main_effect_factor_3",
        "interaction_factor_1_factor_2",
        "interaction_factor_1_factor_3",
        "interaction_factor_2_factor_3",
        "interaction_factor_1_factor_2_factor_3",
    ]

    eff_v = _select_effects_for_param(dv)
    eff_a = _select_effects_for_param(da)

    # Restore previous version: build 14 panels (v 7 + a 7) and auto split to two rings.
    paired = []
    for key in eff_keys:
        ev = next((e for e in eff_v if str(e).startswith(key)), None)
        ea = next((e for e in eff_a if str(e).startswith(key)), None)
        if ev is not None:
            paired.append(("v", ev))
        if ea is not None:
            paired.append(("a", ea))

    # Fill any missing effects
    for e in eff_v:
        if ("v", e) not in paired:
            paired.append(("v", e))
    for e in eff_a:
        if ("a", e) not in paired:
            paired.append(("a", e))
    paired = paired[:14]

    n_total = len(paired)
    n_inner, n_outer = _split_two_rings(n_total)
    inner_specs = paired[:n_inner]
    outer_specs = paired[n_inner : n_inner + n_outer]

    fig_w = FIG_BASE_W * 3.8
    fig_h = FIG_BASE_W * 3.8
    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=False)

    panel_w = 0.160
    panel_h = 0.160
    inner_axes, outer_axes = ring_panel_axes_two_rings(
        fig,
        n_inner=n_inner,
        n_outer=n_outer,
        panel_w=panel_w,
        panel_h=panel_h,
        # inner ring pulled inward by ~1/4 panel, especially vertically
        r_inner_x=0.215,
        r_inner_y=0.200,
        r_outer_x=0.44,
        r_outer_y=0.44,
    )

    def _assign_specs_left_right(axes, specs):
        """Place a on left half and v on right half as much as possible."""
        if not specs:
            return specs

        # Force layout computation so we can read true positions.
        fig.canvas.draw()
        pos = []
        for i, ax in enumerate(axes):
            p = ax.get_position()
            cx = (p.x0 + p.x1) / 2.0
            cy = (p.y0 + p.y1) / 2.0
            pos.append((i, cx, cy))

        left_idx = [i for i, cx, cy in pos if cx < 0.5]
        right_idx = [i for i, cx, cy in pos if cx >= 0.5]

        # top->bottom assignment within each side for visual consistency
        left_idx = sorted(left_idx, key=lambda i: next(cy for j, cx, cy in pos if j == i), reverse=True)
        right_idx = sorted(right_idx, key=lambda i: next(cy for j, cx, cy in pos if j == i), reverse=True)

        a_specs = [s for s in specs if s[0] == "a"]
        v_specs = [s for s in specs if s[0] == "v"]
        other_specs = [s for s in specs if s[0] not in {"a", "v"}]

        assigned = [None] * len(axes)

        # Left: prefer a
        for i in left_idx:
            if a_specs:
                assigned[i] = a_specs.pop(0)
            elif v_specs:
                assigned[i] = v_specs.pop(0)
            elif other_specs:
                assigned[i] = other_specs.pop(0)

        # Right: prefer v
        for i in right_idx:
            if v_specs:
                assigned[i] = v_specs.pop(0)
            elif a_specs:
                assigned[i] = a_specs.pop(0)
            elif other_specs:
                assigned[i] = other_specs.pop(0)

        leftovers = a_specs + v_specs + other_specs
        for i in range(len(assigned)):
            if assigned[i] is None and leftovers:
                assigned[i] = leftovers.pop(0)

        # Return in axis order, truncated to drawn specs length.
        return [assigned[i] for i in range(min(len(specs), len(assigned))) if assigned[i] is not None]

    inner_specs = _assign_specs_left_right(inner_axes, inner_specs)
    outer_specs = _assign_specs_left_right(outer_axes, outer_specs)

    def _draw_delta(ax, par, eff):
        src = dv if par == "v" else da
        sub = src[src["effect"] == eff]
        x = sub["x"].to_numpy(dtype=float)
        y = sub["y"].to_numpy(dtype=float)
        ax.scatter(x, y, s=18, alpha=0.78, color=THEME["delta"], edgecolors="white", linewidths=0.3)
        add_identity_line(ax, x, y)
        ax.grid(alpha=0.2, linewidth=0.35, color=THEME["grid"])
        rr = pearson_r(x, y)
        ax.text(
            0.02,
            0.98,
            rf"$r={rr:.2f}$, $n={len(x)}$",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7.0,
            bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
        )
        ax.set_xlabel(rf"$\Delta_{{true}}^{{({par})}}$", labelpad=1)
        ax.set_ylabel(rf"$\Delta_{{estimated}}^{{({par})}}$", labelpad=1)
        ax.set_title(clean_title(effect_label_from_code(eff), width=18), pad=2)

    for ax, (par, eff) in zip(inner_axes, inner_specs):
        _draw_delta(ax, par, eff)
    for ax, (par, eff) in zip(outer_axes, outer_specs):
        _draw_delta(ax, par, eff)

    # Hide any unused axes (prevents blank frame-only panels)
    for ax in inner_axes[len(inner_specs):]:
        ax.set_visible(False)
    for ax in outer_axes[len(outer_specs):]:
        ax.set_visible(False)

    out_base = os.path.join(RECOVERY_FIG_DIR, f"parameter_recovery_{RECOVERY_FILE_TAG}_delta_scatter_va_two_rings")
    save_pub_figure(fig, out_base)
    plt.close(fig)


Using: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\parameter_recovery\va_vat\parameter_recovery_va_vat_by_rep.csv
Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\results\parameter_recovery\va_vat\parameter_recovery_va_vat_per_cell_correlations.csv


,parameter,cell,n_pairs,true_range,pearson_r,RMSE,nRMSE_pct,CCC,ICC3_1
8,a,Long.prob_diff.flexibility,30,5.776515,0.938619,0.790456,13.683968,0.918820,0.920896
9,a,Long.prob_diff.stability,30,5.337626,0.939537,0.647633,12.133347,0.933660,0.933663
10,a,Long.prob_same.flexibility,30,5.435773,0.971840,0.484475,8.912712,0.961152,0.971072
11,a,Long.prob_same.stability,30,5.255335,0.948506,0.503327,9.577452,0.944348,0.948269
12,a,Short.prob_diff.flexibility,30,5.802380,0.944410,0.576242,9.931130,0.937623,0.937733
13,a,Short.prob_diff.stability,30,5.212507,0.969319,0.441917,8.478021,0.966255,0.969274
14,a,Short.prob_same.flexibility,30,5.408508,0.966658,0.445079,8.229246,0.963340,0.965677
15,a,Short.prob_same.stability,30,5.375626,0.900864,0.849415,15.801228,0.892375,0.892377
16,t,__t__,30,0.149915,0.924230,0.017872,11.921310,0.919525,0.919695
0,v,Long.prob_diff.flexibility,30,5.726250,0.987165,0.335745,5.863259,0.978192,0.978471


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_per_cell_scatter_vat_two_rings.png


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_per_cell_scatter_vat_two_rings.pdf


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


...

Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_delta_scatter_va_two_rings.png


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_delta_scatter_va_two_rings.pdf


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_delta_scatter_va_two_rings.eps


Saved: D:\文件同步\文件\数据分析结果\data_code_figure\hddm_analysis\figures\parameter_recovery\va_vat\parameter_recovery_va_vat_delta_scatter_va_two_rings.svg
